# 第七章：持久化与检查点

## 学习目标
- 理解 LangGraph 的检查点（Checkpoint）机制
- 用 `MemorySaver` 实现内存持久化
- 用 `thread_id` 管理多个独立会话
- 用 `get_state()` 查看当前状态
- 用 `get_state_history()` 实现「时间旅行」
- 用 `update_state()` 手动修改状态

## 0. 环境准备

In [ ]:
from importlib.metadata import version
print(f"langgraph 版本: {version('langgraph')}")

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="qwen-plus",
    openai_api_base=os.environ["Qwen_API_BASE"],
    openai_api_key=os.environ["Qwen_API_KEY"],
)

## 1. 为什么需要持久化？

前面几章构建的图都有一个问题：**每次 `invoke()` 都是从零开始**。对话没有记忆，上一轮说了什么，下一轮完全不知道。先来直观感受这个问题。

### 问题演示：没有记忆的聊天机器人

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import HumanMessage

# 最简单的聊天机器人
def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

# 注意：没有 checkpointer
app = graph.compile()

In [ ]:
# 第一轮对话
result1 = app.invoke({"messages": [HumanMessage(content="我叫小明，请记住我的名字")]})
print("第一轮:", result1["messages"][-1].content)

In [ ]:
# 第二轮对话——模型不记得小明是谁
result2 = app.invoke({"messages": [HumanMessage(content="我叫什么名字？")]})
print("第二轮:", result2["messages"][-1].content)

第二轮调用完全不知道「小明」的存在。每次 `invoke()` 都是独立的，状态不会跨调用保留。

当然，你可以手动把上一轮的 messages 拼进下一轮输入——但这很繁琐，而且图一旦复杂起来（多个节点、工具调用），手动管理状态几乎不可行。

### 解决方案：添加 MemorySaver

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# 创建检查点存储器
checkpointer = MemorySaver()

# 同样的图，编译时加上 checkpointer
app_with_memory = graph.compile(checkpointer=checkpointer)

In [ ]:
# 关键：调用时必须传 config，指定 thread_id
config = {"configurable": {"thread_id": "chat-1"}}

# 第一轮
result1 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="我叫小明，请记住我的名字")]},
    config,
)
print("第一轮:", result1["messages"][-1].content)

In [ ]:
# 第二轮——同一个 thread_id，模型记得小明
result2 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="我叫什么名字？")]},
    config,
)
print("第二轮:", result2["messages"][-1].content)

### 刚才发生了什么？

只改了两处：

| 改动 | 代码 |
|------|------|
| 编译时加 checkpointer | `graph.compile(checkpointer=checkpointer)` |
| 调用时传 config | `app.invoke(input, {"configurable": {"thread_id": "..."}})` |

LangGraph 在每个节点执行完后，自动把当前完整状态保存为一个**检查点（Checkpoint）**。下次用同一个 `thread_id` 调用时，会先恢复上一次的状态，再把新输入追加进去。

所以第二轮调用时，模型看到的 `messages` 实际上是：
1. `HumanMessage("我叫小明...")` — 来自第一轮
2. `AIMessage("好的...")` — 第一轮的回复
3. `HumanMessage("我叫什么名字？")` — 第二轮的新输入

完整的对话历史自动恢复，不用我们手动管理。

## 2. Checkpoint 机制解析

理解了「加 checkpointer 就有记忆」之后，来看看它内部的工作原理。

### 核心概念

- **Checkpoint**：图在某个时间点的完整状态快照（State 的所有字段值）
- **Checkpointer**：负责存储和检索 Checkpoint 的后端
- **Thread**：一条独立的执行线（用 `thread_id` 标识），每个 thread 有自己的 checkpoint 链

### 自动保存的时机

每次一个节点执行完毕，LangGraph 就自动保存一个 checkpoint。如果图有 3 个节点顺序执行，一次 `invoke()` 就会产生 3 个 checkpoint（加上初始状态共 4 个）。

### Checkpoint 链

每个 checkpoint 记录了「父 checkpoint」的引用，形成一条链：

```
初始状态 → 节点A执行后 → 节点B执行后 → 节点C执行后（最新）
```

这条链支持「时间旅行」——可以回溯到任意历史状态。

### Checkpointer 类型

| Checkpointer | 存储位置 | 重启后保留？ | 适用场景 |
|--------------|---------|------------|----------|
| `MemorySaver` | 内存（Python dict） | 否 | 开发、测试 |
| `PostgresSaver` | PostgreSQL | 是 | 生产环境 |
| `SqliteSaver` | SQLite 文件 | 是 | 轻量生产、单机 |

本章用 `MemorySaver` 学习所有概念，它和数据库版的 API 完全一致——换存储后端只需换一行代码。

## 3. 多线程管理

不同的 `thread_id` 对应完全独立的会话。多个用户可以同时使用同一个图，互不影响。

In [ ]:
# 用一个新的 checkpointer，避免前面例子的干扰
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

# Alice 的会话
config_alice = {"configurable": {"thread_id": "user-alice"}}
# Bob 的会话
config_bob = {"configurable": {"thread_id": "user-bob"}}

In [ ]:
# Alice 聊天气
result = app.invoke(
    {"messages": [HumanMessage(content="北京今天天气怎么样？")]},
    config_alice,
)
print("Alice:", result["messages"][-1].content[:80])

In [ ]:
# Bob 聊数学
result = app.invoke(
    {"messages": [HumanMessage(content="1+1 等于几？")]},
    config_bob,
)
print("Bob:", result["messages"][-1].content[:80])

In [ ]:
# Alice 追问——她的上下文是天气，不是数学
result = app.invoke(
    {"messages": [HumanMessage(content="那明天呢？")]},
    config_alice,
)
print("Alice 追问:", result["messages"][-1].content[:80])

In [ ]:
# Bob 追问——他的上下文是数学，不是天气
result = app.invoke(
    {"messages": [HumanMessage(content="那 2+2 呢？")]},
    config_bob,
)
print("Bob 追问:", result["messages"][-1].content[:80])

### 刚才发生了什么？

Alice 和 Bob 用同一个图实例，但各自有独立的对话历史：

- Alice 的 `thread_id` 是 `"user-alice"`，她的上下文全是天气
- Bob 的 `thread_id` 是 `"user-bob"`，他的上下文全是数学

两个线程互不干扰。`thread_id` 是 checkpoint 的主键——不同 thread_id 对应不同的 checkpoint 链。

在实际应用中，`thread_id` 通常对应一个会话 ID（session ID）。一个用户可以有多个 thread（多个对话窗口）。

## 4. 查看状态 — get_state()

`get_state()` 让你检查某个线程的当前状态，不需要触发图的执行。

In [ ]:
# 查看 Alice 的当前状态
state = app.get_state(config_alice)

print(type(state))  # StateSnapshot 对象

In [ ]:
# values：当前状态的所有字段
print(f"消息数量: {len(state.values['messages'])}")
for msg in state.values["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}] {msg.content[:60]}")

In [ ]:
# next：接下来要执行的节点（空元组 = 图已完成）
print(f"next: {state.next}")

# created_at：检查点的创建时间
print(f"created_at: {state.created_at}")

# config：包含 checkpoint_id 的完整配置
print(f"config: {state.config}")

### StateSnapshot 的关键字段

| 字段 | 类型 | 说明 |
|------|------|------|
| `values` | `dict` | 当前状态的所有字段值 |
| `next` | `tuple` | 接下来要执行的节点名。空元组表示图已执行完毕 |
| `config` | `dict` | 包含 `thread_id` 和 `checkpoint_id` 的配置 |
| `created_at` | `str` | 检查点创建时间（ISO 8601） |
| `parent_config` | `dict` | 父检查点的配置（用于历史回溯） |
| `metadata` | `dict` | 元数据（哪个节点写的、步骤号等） |
| `tasks` | `tuple` | 待执行的任务列表 |

`get_state()` 的典型用途：
- 检查对话进行到哪一步了
- 判断图是否执行完毕（`state.next` 是否为空）
- 获取 `checkpoint_id` 用于时间旅行

## 5. 时间旅行 — get_state_history()

这是 LangGraph checkpoint 最强大的能力：你可以回溯到任意历史状态，甚至从历史状态分叉出新的执行路径。

### 准备：创建一段多轮对话

In [ ]:
# 新开一个线程，进行 3 轮对话
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "time-travel-demo"}}

# 第 1 轮
app.invoke({"messages": [HumanMessage(content="我最喜欢的颜色是蓝色")]}, config)
print("第 1 轮完成")

# 第 2 轮
app.invoke({"messages": [HumanMessage(content="我最喜欢的动物是猫")]}, config)
print("第 2 轮完成")

# 第 3 轮
app.invoke({"messages": [HumanMessage(content="总结一下我告诉你的信息")]}, config)
print("第 3 轮完成")

### 查看所有检查点

In [ ]:
# get_state_history 返回所有检查点，从最新到最旧
history = list(app.get_state_history(config))

print(f"共 {len(history)} 个检查点\n")

for i, snapshot in enumerate(history):
    n_msgs = len(snapshot.values.get("messages", []))
    next_nodes = snapshot.next
    checkpoint_id = snapshot.config["configurable"]["checkpoint_id"]
    print(f"[{i}] checkpoint_id={checkpoint_id[:12]}...  "
          f"消息数={n_msgs}  next={next_nodes}")

### 理解检查点时间线

3 轮对话 × 每轮 1 个节点 = 图执行了 3 次，但检查点数量更多。每次 `invoke()` 会在以下时刻保存：

1. 接收到输入时（`__start__` 节点之后）
2. `chatbot` 节点执行后

所以 3 轮对话大约产生 6+ 个检查点。有些检查点的 `next` 不为空，表示那个时刻图还没执行完（正等待下一个节点）。

### 回到过去：用历史 checkpoint 的 config 调用

In [ ]:
# 找到第 1 轮结束时的检查点（有 2 条消息：Human + AI）
target = None
for snapshot in history:
    msgs = snapshot.values.get("messages", [])
    # 第 1 轮结束 = 2 条消息，且图已完成（next 为空）
    if len(msgs) == 2 and not snapshot.next:
        target = snapshot
        break

print(f"找到目标检查点，消息数: {len(target.values['messages'])}")
for msg in target.values["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}] {msg.content[:60]}")

In [ ]:
# 用这个历史检查点的 config 发起新调用——相当于「从第 1 轮之后分叉」
# 此时模型只知道蓝色，不知道猫
forked_result = app.invoke(
    {"messages": [HumanMessage(content="我告诉过你什么信息？")]},
    target.config,  # 使用历史检查点的 config
)
print("分叉后的回答:", forked_result["messages"][-1].content)

### 刚才发生了什么？

我们用第 1 轮结束时的 `config`（包含那个时间点的 `checkpoint_id`）发起了新调用。此时：

- 模型只看到第 1 轮的对话（蓝色），不知道第 2 轮的「猫」
- 新调用从历史状态「分叉」出来，不影响原来的对话线

这就是「时间旅行」：

```
第1轮 → 第2轮(猫) → 第3轮(总结)    ← 原始路径
  └───→ 分叉("我告诉过你什么？")    ← 新路径
```

使用场景：
- **调试**：回到出错之前的状态，换个输入重试
- **撤销**：用户觉得上一轮回答不好，退回重新来
- **A/B 测试**：从同一个状态分叉，比较不同路径的效果

## 6. 手动修改状态 — update_state()

有时你需要直接修改某个线程的状态，而不是通过图的执行来更新。比如：
- 模型说了不正确的信息，需要人工纠正
- 想注入一条系统消息
- 需要重置某个字段的值

In [ ]:
# 新线程，先对话一轮
checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "update-demo"}}

result = app.invoke(
    {"messages": [HumanMessage(content="1+1 等于几？")]},
    config,
)
print("原始回答:", result["messages"][-1].content)

In [ ]:
# 查看当前状态
state = app.get_state(config)
print(f"修改前消息数: {len(state.values['messages'])}")
for msg in state.values["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}] {msg.content[:60]}")

In [ ]:
from langchain_core.messages import AIMessage

# 手动添加一条 AI 消息（模拟纠正模型回答）
corrected = AIMessage(content="1+1 等于 2。这是基础算术。")

# update_state：手动写入状态
# as_node 指定这次更新是以哪个节点的身份写入的
new_config = app.update_state(
    config,
    values={"messages": [corrected]},
    as_node="chatbot",
)
print(f"返回的新 config: {new_config['configurable']['checkpoint_id'][:12]}...")

In [ ]:
# 查看修改后的状态
state = app.get_state(config)
print(f"修改后消息数: {len(state.values['messages'])}")
for msg in state.values["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}] {msg.content[:60]}")

In [ ]:
# 后续对话会基于修改后的状态继续
result = app.invoke(
    {"messages": [HumanMessage(content="那 2+2 呢？")]},
    config,
)
print("后续回答:", result["messages"][-1].content)

### update_state 详解

```python
new_config = graph.update_state(
    config,          # 要修改哪个线程/检查点
    values={...},    # 要写入的值
    as_node="...",   # 以哪个节点的身份写入
)
```

**`as_node` 参数的作用**：

LangGraph 需要知道这次更新是「哪个节点产生的」，因为：
1. `values` 会经过该节点对应字段的 reducer 处理（追加 vs 覆盖）
2. `state.next` 会据此更新——图会从这个节点的下一步继续

比如我们用 `as_node="chatbot"` 写入了一条消息：
- `messages` 字段使用的是 `add_messages` reducer，所以新消息被**追加**而非覆盖
- 由于 `chatbot` 是最后一个节点（之后就是 END），所以 `next` 为空

| 参数 | 必需？ | 说明 |
|------|--------|------|
| `config` | 是 | 指定线程和/或检查点 |
| `values` | 是 | 要更新的字段值 |
| `as_node` | 否（推荐写） | 以哪个节点的身份写入，影响 reducer 行为和 next 计算 |

### 替换消息 vs 追加消息

`MessagesState` 使用的 `add_messages` reducer 有一个特性：如果新消息的 `id` 和已有消息相同，会**替换**而非追加。利用这个特性可以修改已有消息：

In [ ]:
# 查看当前最后一条 AI 消息的 id
state = app.get_state(config)
last_ai_msg = [m for m in state.values["messages"] if isinstance(m, AIMessage)][-1]
print(f"最后一条 AI 消息 id: {last_ai_msg.id}")
print(f"内容: {last_ai_msg.content[:60]}")

In [ ]:
# 用相同 id 创建修改后的消息——会替换而非追加
replaced = AIMessage(
    content="2+2=4，这也是基础算术。",
    id=last_ai_msg.id,  # 关键：相同 id
)

app.update_state(config, values={"messages": [replaced]}, as_node="chatbot")

# 验证：消息被替换了，总数没变
state = app.get_state(config)
print(f"消息总数: {len(state.values['messages'])}")
last_ai = [m for m in state.values["messages"] if isinstance(m, AIMessage)][-1]
print(f"替换后内容: {last_ai.content}")

| 操作 | 做法 |
|------|------|
| 追加新消息 | `update_state(config, {"messages": [AIMessage(content="...")]})` |
| 替换已有消息 | `update_state(config, {"messages": [AIMessage(content="...", id=原消息.id)]})` |

## 7. 实战——可恢复的多轮对话 Agent

把本章学到的所有东西组合起来：构建一个带工具的 ReAct Agent，支持多轮对话、状态检查和时间旅行。

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 定义工具
@tool
def search_knowledge(query: str) -> str:
    """搜索知识库获取信息"""
    # 模拟知识库
    kb = {
        "python": "Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年发布。",
        "langgraph": "LangGraph 是一个用于构建有状态 LLM 应用的框架，支持循环和分支。",
        "langchain": "LangChain 是一个 LLM 应用开发框架，提供模型、提示词、链等抽象。",
    }
    for key, value in kb.items():
        if key in query.lower():
            return value
    return f"未找到关于 '{query}' 的信息"

@tool
def calculator(expression: str) -> str:
    """计算数学表达式"""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"计算错误: {e}"

In [ ]:
# 创建带 checkpointer 的 Agent
checkpointer = MemorySaver()

agent = create_react_agent(
    llm,
    tools=[search_knowledge, calculator],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "agent-demo"}}

In [ ]:
# 第 1 轮：搜索知识库
result = agent.invoke(
    {"messages": [HumanMessage(content="帮我查一下 LangGraph 是什么")]},
    config,
)
print("第 1 轮:", result["messages"][-1].content[:100])

In [ ]:
# 第 2 轮：引用上一轮结果 + 使用计算器
result = agent.invoke(
    {"messages": [HumanMessage(content="它是哪一年之后的框架？帮我算一下到 2026 年过了多少年")]},
    config,
)
print("第 2 轮:", result["messages"][-1].content[:100])

In [ ]:
# 查看 Agent 的完整状态
state = agent.get_state(config)
print(f"总消息数: {len(state.values['messages'])}")
print(f"图已完成: {state.next == ()}")
print()
# 展示消息流
for msg in state.values["messages"]:
    role = msg.__class__.__name__.replace("Message", "")
    content = msg.content[:80] if msg.content else "(无文本内容)"
    extra = ""
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        extra = f" -> 调用工具: {[tc['name'] for tc in msg.tool_calls]}"
    print(f"  [{role}] {content}{extra}")

In [ ]:
# 时间旅行：查看所有检查点
history = list(agent.get_state_history(config))
print(f"检查点总数: {len(history)}")
print()

# 找到第 1 轮结束时的状态（消息较少且 next 为空）
for snapshot in history:
    msgs = snapshot.values.get("messages", [])
    if not snapshot.next and 2 <= len(msgs) <= 5:
        print(f"候选检查点: {len(msgs)} 条消息")
        for m in msgs:
            role = m.__class__.__name__.replace("Message", "")
            print(f"  [{role}] {m.content[:50]}")
        print()

### 关于生产环境的持久化

`MemorySaver` 把所有数据存在 Python 进程的内存中。这意味着：
- 进程重启后数据全部丢失
- 不能跨进程共享（多个服务实例无法共享状态）

生产环境应使用数据库后端：

```python
# PostgreSQL（推荐，支持高并发）
from langgraph.checkpoint.postgres import PostgresSaver
checkpointer = PostgresSaver.from_conn_string("postgresql://...")

# SQLite（轻量方案）
from langgraph.checkpoint.sqlite import SqliteSaver
checkpointer = SqliteSaver.from_conn_string("checkpoints.db")
```

API 完全一致——`get_state()`、`get_state_history()`、`update_state()` 用法不变，只是存储后端不同。

## 8. 常见问题

### Q: 报错 "thread_id is required" 或类似错误

编译时加了 `checkpointer`，调用时就**必须**传 `config`：

```python
# 错误：忘了传 config
app.invoke({"messages": [HumanMessage(content="你好")]})

# 正确
app.invoke(
    {"messages": [HumanMessage(content="你好")]},
    {"configurable": {"thread_id": "my-thread"}},
)
```

### Q: MemorySaver 重启后数据丢失了？

正常现象。`MemorySaver` 是内存存储，进程结束数据就没了。生产环境用 `PostgresSaver` 或 `SqliteSaver`。

### Q: 不同图实例能共享检查点吗？

可以，只要：
1. 使用同一个 `checkpointer` 实例
2. 两个图的 State schema 兼容（字段名和类型一致）

```python
checkpointer = MemorySaver()
app_v1 = graph_v1.compile(checkpointer=checkpointer)
app_v2 = graph_v2.compile(checkpointer=checkpointer)
# 同一个 thread_id 的数据是共享的
```

### Q: 对话历史越来越长，会不会有性能问题？

会。随着对话轮次增加：
- 每个 checkpoint 保存完整 `messages` 列表，占用存储空间
- 发给 LLM 的消息越来越长，增加延迟和成本

常见解决方案：
- 限制消息窗口（只保留最近 N 条）
- 周期性生成摘要，替代历史消息
- 在节点中手动裁剪 `messages` 列表

## 总结

本章学习了：
- 没有 checkpointer 时，每次 `invoke()` 都是从零开始，没有对话记忆
- `MemorySaver` + `thread_id` 两步实现持久化，自动保存和恢复状态
- 不同 `thread_id` 对应完全独立的会话
- `get_state()` 检查任意线程的当前状态
- `get_state_history()` 遍历所有历史检查点，实现「时间旅行」
- `update_state()` 手动修改状态，支持追加和替换消息
- 生产环境用 `PostgresSaver` / `SqliteSaver` 替代 `MemorySaver`

## 下一章预告

**第八章：生产实践（流式输出、调试、部署）** —— 最后一章，学习将 LangGraph 应用推向生产环境所需的关键技术：流式输出让用户不用等、调试工具帮你定位问题、部署方案让应用上线。